# ES-LLM: Layer-wise Fine-Tuning mit Evolutionary Strategies

Dieses Notebook führt dich durch den kompletten Workflow:
1. **Setup** – Repo klonen, Dependencies installieren
2. **Inspect** – Modell-Architektur inspizieren
3. **Baseline** – Unveränderte Accuracy messen
4. **Train** – ES-Training auf ausgewählten Layern
5. **Evaluate** – Fine-tuned Modell testen
6. **Export** – Ergebnisse nach Google Drive speichern

**GPU-Empfehlung:** Runtime → Change runtime type → **T4 GPU** (kostenlos) oder **A100** (Colab Pro)

## 0. GPU prüfen

In [ ]:
import torch
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Keine GPU! Gehe zu Runtime → Change runtime type → GPU")

## 1. Setup – Repo klonen & Dependencies installieren

In [ ]:
# ── Repo klonen ──
import os

REPO_URL = "https://github.com/Haso001/PG_ML_AI.git"
REPO_DIR = "/content/PG_ML_AI"
PROJECT_DIR = f"{REPO_DIR}/es-llm-finetune-Hasan-Dev"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Repo existiert → neueste Änderungen pullen
    !cd {REPO_DIR} && git pull
    print(f"{REPO_DIR} aktualisiert (git pull).")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Alternativ: Repo als ZIP hochladen ──
# Falls du kein Git-Repo hast, lade den Ordner als ZIP hoch:
#
# from google.colab import files
# uploaded = files.upload()  # ZIP-Datei hochladen
# !unzip -o es-llm-finetune.zip -d /content/es-llm-finetune
# os.chdir("/content/es-llm-finetune")

In [ ]:
# ── Dependencies installieren ──
!pip install -q torch transformers datasets accelerate pyyaml cmaes

# Verifizieren
import torch, transformers
print(f"torch={torch.__version__}, transformers={transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── src/ zum Python-Pfad hinzufügen ──
import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")

# Import-Test
from es_llm.model.loader import load_model_and_tokenizer, inspect_model_layers
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness
from es_llm.training.es_trainer import train
from es_llm.utils.config import load_config
print("Alle Imports erfolgreich!")

## 2. Modell-Architektur inspizieren

Zuerst schauen wir uns an, welche Layer das Qwen2.5-0.5B hat und wie viele Parameter pro Layer existieren.

In [ ]:
# Modell laden (float16 auf GPU → ~1 GB VRAM)
model, tokenizer = load_model_and_tokenizer(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",
    device="cuda",
)

total = sum(p.numel() for p in model.parameters())
print(f"\nGesamt: {total:,} Parameter ({total/1e6:.1f}M)")
print(f"VRAM belegt: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Alle Layer auflisten
layers = inspect_model_layers(model)

# Per-Layer Zusammenfassung
from collections import defaultdict
per_layer = defaultdict(int)
for l in layers:
    parts = l["name"].split(".")
    key = "other"
    for i, p in enumerate(parts):
        if p == "layers" and i + 1 < len(parts) and parts[i + 1].isdigit():
            key = f"layers.{parts[i + 1]}"
            break
    per_layer[key] += l["numel"]

print(f"{'Layer':<20s} {'Parameter':>15s}")
print("─" * 37)
for key in sorted(per_layer, key=lambda k: (not k.startswith("layers"), k)):
    print(f"{key:<20s} {per_layer[key]:>15,}")

In [ ]:
# Detail: Layer 23 (letzter Decoder-Block)
print("\nLayer 23 – Detail:")
print(f"{'Name':<55s} {'Shape':<25s} {'Params':>10s}")
print("─" * 92)
for l in layers:
    if "layers.23" in l["name"]:
        print(f"{l['name']:<55s} {str(l['shape']):<25s} {l['numel']:>10,}")

## 3. Layer-Selektion testen

Wir nutzen den **kompletten letzten Decoder-Block** (Layer 23) mit allen Komponenten:
- **Attention**: q_proj, k_proj, v_proj, o_proj
- **MLP**: gate_proj, up_proj, down_proj
- **LayerNorm**: input_layernorm, post_attention_layernorm

Das ergibt **~14.9M Parameter** als einen einzigen Vektor θ, den ES optimiert.

In [ ]:
# Kompletter letzter Layer → alle ~14.9M Parameter (Attention + MLP + LayerNorm)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="all",
)
print(selector.summary())
print(f"\nGesamt: {selector.num_target_elements:,} Parameter als 1-D Vektor θ")

In [ ]:
# MLP des letzten Layers → mehr Parameter
selector_mlp = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="mlp",
)
print(selector_mlp.summary())

In [ ]:
# Attention Q/K/V der letzten 2 Layer
selector_attn = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[22, 23],
    components="attention_qkv",
)
print(selector_attn.summary())

## 4. Baseline messen

Wie gut ist das Modell **unverändert** auf GSM8K?

In [ ]:
# Baseline-Fitness (kleines Subset zum schnellen Testen)
fitness_fn = GSM8KFitness(
    num_samples=30,       # Anzahl Fragen
    split="test",
    max_new_tokens=256,
)

baseline_acc = fitness_fn.evaluate(model, tokenizer)
print(f"\n✅ Baseline Accuracy (GSM8K test, n=30): {baseline_acc:.2%}")

## 5. ES-Training starten

### Option A: Mit YAML-Config (empfohlen)

In [ ]:
# Config laden und starten
cfg = load_config(f"{PROJECT_DIR}/configs/gsm8k_last_layer.yaml")

# Config anzeigen
import json
print(json.dumps(cfg, indent=2))

In [ ]:
# Optional: Werte überschreiben für schnellen Test mit vollem Layer
cfg["layer_selection"]["layer_indices"] = [23]
cfg["layer_selection"]["components"] = "all"   # alle ~14.9M Params

cfg["es"]["num_generations"] = 10              # Schneller Testlauf
cfg["es"]["population_size"] = 10
cfg["es"]["sigma"] = 0.001                     # Klein weil 14.9M Dimensionen
cfg["es"]["learning_rate"] = 0.001

cfg["fitness"]["num_samples"] = 15             # Samples pro Fitness-Eval
cfg["fitness"]["max_new_tokens"] = 128

cfg["output"]["dir"] = "/content/experiments/runs"

# ACHTUNG: Bei Option A wird das Modell NEU geladen.
# Vorheriges Modell freigeben, um VRAM zu sparen:
del model, tokenizer
torch.cuda.empty_cache()

# Training starten!
run_dir = train(cfg)
print(f"\n✅ Training abgeschlossen! Ergebnisse in: {run_dir}")

### Option B: Manueller Training-Loop (mehr Kontrolle)

Zwei Varianten:
- **B1: Accuracy-Fitness** – generiert Antworten, prüft ob richtig (langsam, diskret)
- **B2: Log-Likelihood-Fitness** – misst Wahrscheinlichkeit der richtigen Antwort (schnell, stetig)

**Empfehlung: B2 (Log-Likelihood)** → ~10× schneller, besseres Signal für ES

In [ ]:
# ── Option B1: Manueller ES-Loop mit ACCURACY-Fitness (langsam, diskret) ──
# (Gleicher Ansatz wie der erste Run → ~10 Min./Generation)

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],       # Letzter Decoder-Block
    components="all",         # Attention + MLP + LayerNorm → ~14.9M Params
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus
es = OpenAIES(
    population_size=200,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Accuracy – langsam)
fitness_fn = GSM8KFitness(num_samples=15, split="train", max_new_tokens=128)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 10

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.3f}  mean={result.mean_fitness:.3f}")

print(f"\n✅ Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

In [ ]:
# ── Option B2: Manueller ES-Loop mit LOG-LIKELIHOOD-Fitness (schnell, stetig) ──
# ~10x schneller als B1 → ~1 Min./Generation statt ~10 Min.

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswaehlen - KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="all",
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus (gleiche Settings wie erster Run)
es = OpenAIES(
    population_size=10,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Log-Likelihood - schnell und stetig!)
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=30,        # mehr Samples moeglich weil viel schneller
    split="train",
    target_mode="short",   # nur "#### <number>" bewerten
)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 10

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.4f}  mean={result.mean_fitness:.4f}")

print(f"\nLL-Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

### Option C: Optimiertes ES-Training v3 für A100/H100 (empfohlen)

Basierend auf **Salimans et al. 2017** – "Evolution Strategies as a Scalable Alternative to RL"

**v3-Fixes gegenüber v2:**

| Problem in v2 | Fix in v3 |
|---|---|
| **Reshuffling pro Kandidat** → jeder Kandidat sah andere Fragen, ES-Gradient war Rauschen | Reshuffle pro **Generation**: alle Kandidaten sehen gleiche Fragen |
| Training stagnierte ab Gen 70, lief aber bis 150 | `num_generations: 80` → kein verschwendetes Compute |
| σ sank auf 0.0008 → Exploration tot | `sigma_final: 0.0015` → bleibt lebensfähig |
| Tokenisierung 96× pro Gen (jeder Kandidat neu) | 1× pro Gen → **massiver Speedup** |

**Kernfeatures:**
- **Full-Reasoning Target**: LL über gesamte Step-by-Step Lösung
- **Data Reshuffling per Generation**: Alle 96 Kandidaten der gleichen Gen → gleiche 100 Fragen. Nächste Gen → neue 100 Fragen aus Pool
- **Adam-Optimizer + Cosine σ-Decay**
- **Mittlere Layers [20,21]** statt letzter → kein Catastrophic Forgetting

**Daten-Trennung:** Training auf `train`-Split (7473), Evaluation auf `test`-Split (1319)

**Erwartete Laufzeit:** ~30-45 Min. auf A100

In [ ]:
# ── Option C: Optimiertes ES-Training v3 (Salimans et al. 2017) ──
# BUGFIX: Reshuffle pro Generation (nicht pro Kandidat!)
# Budget: ~30-45 Min. auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – Attention von Layer 20+21 (~3.6M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[20, 21],
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3) ES mit Adam-Optimizer + Cosine Sigma-Schedule
NUM_GENERATIONS = 80

es = OpenAIES(
    population_size=96,
    sigma=0.004,                      # Mehr Exploration am Anfang
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.003,               # Weniger aggressiv als v2
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.0015,               # Höher → Exploration bleibt aktiv
    max_generations=NUM_GENERATIONS,
)

# 4) Fitness: Full-Reasoning LL mit per-Generation Reshuffling
#    WICHTIG: reshuffle_data() wird pro Generation aufgerufen (nicht pro Kandidat!)
#    → Alle 96 Kandidaten in einer Gen sehen die GLEICHEN 100 Fragen
#    → Nächste Generation bekommt ANDERE 100 Fragen aus dem Pool
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=100,
    split="train",
    target_mode="full",
    batch_size=8,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print(f"\n{'='*65}")
print(f"  ES-Training v3: {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (β1={es.adam_beta1}, β2={es.adam_beta2})")
print(f"  σ-Schedule: cosine {es.sigma:.4f} → {es.sigma_final:.4f}")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 20+21 Attention)")
print(f"  Fitness: Full-Reasoning LL (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Data: Reshuffle per GENERATION – 100 gleiche Fragen für alle Kandidaten")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print(f"{'='*65}\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # WICHTIG: Neue Fragen für diese Generation (alle Kandidaten sehen die gleichen!)
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever,
        "sigma": es.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
              f"σ={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
              f"ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\n✅ Training abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"   Best-ever LL Fitness: {best_ever:.4f}")

In [ ]:
# ── Vergleich: Fine-tuned (Option C v3) vs Base-Modell auf GSM8K TEST-Split ──
# WICHTIG: Evaluation auf TEST-Split (1319 Fragen) – komplett andere Daten als Training!
# Training nutzte TRAIN-Split (7473 Fragen) → kein Data Leakage

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200  # 200 Fragen aus dem TEST-Split

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell (Option C v3) auf {NUM_EVAL} GSM8K-TEST-Fragen...")
print(f"  (Training war auf TRAIN-Split, Eval ist auf TEST-Split → faire Bewertung)")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*60}")
print(f"  EVALUATION AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"  (Training war auf TRAIN-Split → kein Data Leakage)")
print(f"{'='*60}")
print(f"  Base-Modell:      {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned (v3):  {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:        {diff:+6.2%}")
print(f"{'='*60}")

In [ ]:
# ── Visualisierung: Training + Accuracy-Vergleich ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# ── Plot 1: Fitness über Generationen ──
ax1 = fig.add_subplot(gs[0, 0])
gens = [h["gen"] for h in history]
bests = [h["best"] for h in history]
means = [h["mean"] for h in history]
best_evers = [h["best_ever"] for h in history]

ax1.plot(gens, bests, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens, means, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens, best_evers, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood Fitness")
ax1.set_title("ES Training v3 – Fitness über Generationen\n(Reshuffle-Bugfix + Cosine σ)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Sigma-Schedule ──
ax2 = fig.add_subplot(gs[0, 1])
sigmas = [h["sigma"] for h in history]
ax2.plot(gens, sigmas, linewidth=2, color="#9C27B0")
ax2.set_xlabel("Generation")
ax2.set_ylabel("σ (Noise Std)")
ax2.set_title("Cosine σ-Decay Schedule")
ax2.grid(True, alpha=0.3)
ax2.fill_between(gens, sigmas, alpha=0.15, color="#9C27B0")

# ── Plot 3: Accuracy-Vergleich (Balkendiagramm) ──
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Base-Modell\n(unverändert)", "Fine-tuned v3\n(ES + Full LL + Reshuffle)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax3.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)

ax3.set_ylabel("Accuracy")
ax3.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})\nTrain auf train-Split, Eval auf test-Split")
ax3.set_ylim(0, max(accs) * 1.3)
ax3.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, label="Baseline")
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9)

# ── Plot 4: Zeitverlauf pro Generation ──
ax4 = fig.add_subplot(gs[1, 1])
times = [h["elapsed"] for h in history]
ax4.plot(gens, times, linewidth=1.5, color="#FF5722", alpha=0.7)
ax4.axhline(y=np.mean(times), color="gray", linestyle="--", alpha=0.5,
            label=f"Ø {np.mean(times):.1f}s/gen")
ax4.set_xlabel("Generation")
ax4.set_ylabel("Zeit (Sekunden)")
ax4.set_title("Laufzeit pro Generation")
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

fig.suptitle("ES-Training v3 – Cosine σ + Full Reasoning + Reshuffle + Layer 20/21",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
total_time = sum(times)
print(f"\n{'='*65}")
print(f"  ZUSAMMENFASSUNG – ES-Training v3")
print(f"{'='*65}")
print(f"  Generationen:      {len(history)}")
print(f"  Population:        {es.population_size}")
print(f"  Parameter:         {selector.num_target_elements:,} (Layer 20+21 Attention)")
print(f"  Gesamtzeit:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Ø pro Generation:  {np.mean(times):.1f}s")
print(f"  σ Start → Ende:   {history[0]['sigma']:.5f} → {history[-1]['sigma']:.5f}")
print(f"  Best-Ever LL:      {history[-1]['best_ever']:.4f}")
print(f"  Data Reshuffling:  ON (100 neue Fragen/Gen aus Pool von 5000)")
print(f"  Target Mode:       full (gesamtes Reasoning)")

### Option D: ES-Training v5 – Breitere Layer-Abdeckung (empfohlen nach v3)

Aufbauend auf v3 (+5%) — doppelt so viele Layers für mehr Kapazität.

| | v3 | v5 |
|---|---|---|
| **Layers** | [20, 21] ~3.6M Params | [18, 19, 20, 21] ~7.2M Params |
| **σ** | 0.004 → 0.0015 | 0.003 → 0.001 (kleiner: mehr Params) |
| **LR** | 0.001 | 0.0008 (konservativer) |
| **Generationen** | 80 | 100 |
| **Samples/Gen** | 100 | 120 |
| **Batch-Size** | 8 | 12 (A100 hat genug VRAM) |
| **Laufzeit** | ~60 min | ~75 min |

**Warum 4 Layers?** v3 mit 2 Layers hat +5% erreicht — möglicherweise Kapazitäts-Limit. Layers 18+19 sind sicher (kein Catastrophic Forgetting).

In [ ]:
# ── Option D: ES-Training v5 – 4 Layers (Salimans et al. 2017) ──
# Doppelte Layer-Abdeckung: [18,19,20,21] → ~7.2M Parameter
# Budget: ~75 Min. auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – Attention von Layer 18-21 (~7.2M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[18, 19, 20, 21],
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3) ES mit Adam-Optimizer + Cosine Sigma-Schedule
NUM_GENERATIONS = 100

es = OpenAIES(
    population_size=96,
    sigma=0.003,                      # Kleiner als v3: mehr Params braucht weniger Noise
    learning_rate=0.0008,             # Konservativer bei mehr Params
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.002,               # Leichter: mehr Params braucht weniger Druck
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.001,                # Kann tiefer → mehr Kapazität zum Exploitieren
    max_generations=NUM_GENERATIONS,
)

# 4) Fitness: Full-Reasoning LL mit per-Generation Reshuffling
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=120,                  # Mehr Samples → stabilerer Gradient
    split="train",
    target_mode="full",
    batch_size=12,                    # A100: 12 passt locker bei 0.5B-Modell
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print(f"\n{'='*65}")
print(f"  ES-Training v5: {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (β1={es.adam_beta1}, β2={es.adam_beta2})")
print(f"  σ-Schedule: cosine {es.sigma:.4f} → {es.sigma_final:.4f}")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 18-21 Attention)")
print(f"  Fitness: Full-Reasoning LL (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Data: Reshuffle per GENERATION – 120 gleiche Fragen für alle Kandidaten")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print(f"{'='*65}\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # Neue Fragen für diese Generation
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever,
        "sigma": es.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
              f"σ={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
              f"ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\n✅ Training abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"   Best-ever LL Fitness: {best_ever:.4f}")

In [ ]:
# ── Vergleich: Fine-tuned (Option D v5) vs Base-Modell auf GSM8K TEST-Split ──

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200

# --- 1) Fine-tuned Modell evaluieren ---
print(f"Evaluiere FINE-TUNED Modell (Option D v5) auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*60}")
print(f"  EVALUATION AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"{'='*60}")
print(f"  Base-Modell:      {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned (v5):  {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:        {diff:+6.2%}")
print(f"{'='*60}")

### Option D++: Sequential Fine-Tuning (2 Phasen) – Attention dann MLP

Zuerst Attention [18-21] optimieren, dann auf diesem stabilen State die MLP [21] optimieren.
Das verhindert Catastrophic Forgetting und balanciert Context-Integration mit Wissensupdate.

| Phase | Komponente | Layers | Params | Generationen | σ | Laufzeit |
|---|---|---|---|---|---|---|
| **1** | Attention | [18-21] | ~7.2M | 60 | 0.003→0.001 | ~50 min (A100) |
| **2** | MLP | [21] | ~12M | 60 | 0.001→0.0005 | ~60 min (A100) |
| **Total** | Combined | [18-21] Attn + [21] MLP | ~19.2M | 120 | – | ~110 min |

**Vorteil:** Phase 1 stärkt Context-Integration (Attention) ohne Störung. Phase 2 dann Faktenwissen (MLP) auf stabilem Grund.
**Risiko:** Phase 2 könnte Phase 1 partiell überschreiben → daher kleineres σ in Phase 2.


In [ ]:
# ── Option D++: Phase 1 – ES-Training für Attention [18-21] ──
# Ziel: Context-Integration verbessern (Attention stärken)
# Budget: ~50 Min auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

print("\n" + "="*70)
print("  PHASE 1: ES-Training für Attention [18-21] (Context-Integration)")
print("="*70)

# 1) Base-Modell laden (neu, unbefestigt)
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – NUR Attention [18-21] (~7.2M Parameter)
selector_p1 = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[18, 19, 20, 21],
    components="attention",
)
print(f"\nTarget Phase 1: {selector_p1.num_target_elements:,} Parameter (Attention [18-21])")
print(selector_p1.summary())

# 3) ES-Algorithmus für Phase 1
NUM_GEN_P1 = 60

es_p1 = OpenAIES(
    population_size=96,
    sigma=0.003,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.002,
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.001,
    max_generations=NUM_GEN_P1,
)

# 4) Fitness-Funktion für Phase 1
fitness_fn_p1 = GSM8KLogLikelihoodFitness(
    num_samples=100,
    split="train",
    target_mode="full",
    batch_size=8,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop Phase 1
current_params_p1 = selector_p1.get_flat_params().clone()
best_ever_p1 = -float("inf")
best_ever_params_p1 = current_params_p1.clone()
history_p1 = []

print(f"\n{'='*70}")
print(f"  Phase 1 Start: Attention [18-21] | σ={es_p1.sigma:.4f}")
print(f"{'='*70}\n")

t_total_p1 = time.time()

for gen in range(1, NUM_GEN_P1 + 1):
    t0 = time.time()
    fitness_fn_p1.reshuffle_data()

    candidates = es_p1.ask(current_params_p1)
    fitnesses = []
    for cand in candidates:
        selector_p1.set_flat_params(cand)
        fit = fitness_fn_p1.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es_p1.tell(current_params_p1, candidates, fitnesses)
    current_params_p1 = result.new_params.clone()
    selector_p1.set_flat_params(current_params_p1)

    if result.best_fitness > best_ever_p1:
        best_ever_p1 = result.best_fitness
        best_ever_params_p1 = current_params_p1.clone()

    elapsed = time.time() - t0
    history_p1.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever_p1,
        "sigma": es_p1.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total_p1
        eta = (total_t / gen) * (NUM_GEN_P1 - gen)
        print(f"P1 Gen {gen:3d}/{NUM_GEN_P1} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever_p1:.4f}  "
              f"σ={es_p1.current_sigma:.5f}  [{elapsed:.1f}s/gen, ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Phase 1 abschließen
selector_p1.set_flat_params(best_ever_params_p1)
total_time_p1 = time.time() - t_total_p1
print(f"\n✅ PHASE 1 abgeschlossen ({total_time_p1/60:.1f} min)")
print(f"   Best-ever LL: {best_ever_p1:.4f}")

# ── SPEICHERN: Model nach Phase 1 ──
model_state_p1 = {name: p.data.clone() for name, p in model.named_parameters()}
torch.save(model_state_p1, "/content/model_after_phase1.pt")
print(f"\n📦 Model nach Phase 1 gespeichert: /content/model_after_phase1.pt")

# ──────────────────────────────────────────────────────────────────
# PHASE 2: MLP [21] auf gespeichertem Phase-1-State
# ──────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("  PHASE 2: ES-Training für MLP [21] (Faktenwissen)")
print("  → Startet von Phase-1-State (Attention bereits optimiert)")
print("="*70)

# 1) LADE Phase-1-Model
phase1_state = torch.load("/content/model_after_phase1.pt", weights_only=False)
for name, param in model.named_parameters():
    if name in phase1_state:
        param.data.copy_(phase1_state[name])
print(f"\n✅ Phase-1-State geladen ins Modell")

# 2) Layer auswählen – NUR MLP [21] (~12M Parameter)
selector_p2 = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[21],
    components="mlp",
)
print(f"\nTarget Phase 2: {selector_p2.num_target_elements:,} Parameter (MLP [21])")
print(selector_p2.summary())

# 3) ES-Algorithmus für Phase 2 (konservativer, kleineres σ)
NUM_GEN_P2 = 60

es_p2 = OpenAIES(
    population_size=96,
    sigma=0.001,                    # Kleiner als Phase 1: vorsichtig auf Phase-1-State
    learning_rate=0.0005,           # Auch konservativer
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.001,
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.0005,
    max_generations=NUM_GEN_P2,
)

# 4) Fitness-Funktion für Phase 2 (neuer Pool-Shuffle)
fitness_fn_p2 = GSM8KLogLikelihoodFitness(
    num_samples=120,
    split="train",
    target_mode="full",
    batch_size=12,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop Phase 2
current_params_p2 = selector_p2.get_flat_params().clone()
best_ever_p2 = -float("inf")
best_ever_params_p2 = current_params_p2.clone()
history_p2 = []

print(f"\n{'='*70}")
print(f"  Phase 2 Start: MLP [21] | σ={es_p2.sigma:.6f}")
print(f"  (Aufbauend auf Phase 1 – Attention [18-21] bleibt gefestigt)")
print(f"{'='*70}\n")

t_total_p2 = time.time()

for gen in range(1, NUM_GEN_P2 + 1):
    t0 = time.time()
    fitness_fn_p2.reshuffle_data()

    candidates = es_p2.ask(current_params_p2)
    fitnesses = []
    for cand in candidates:
        selector_p2.set_flat_params(cand)
        fit = fitness_fn_p2.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es_p2.tell(current_params_p2, candidates, fitnesses)
    current_params_p2 = result.new_params.clone()
    selector_p2.set_flat_params(current_params_p2)

    if result.best_fitness > best_ever_p2:
        best_ever_p2 = result.best_fitness
        best_ever_params_p2 = current_params_p2.clone()

    elapsed = time.time() - t0
    history_p2.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever_p2,
        "sigma": es_p2.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total_p2
        eta = (total_t / gen) * (NUM_GEN_P2 - gen)
        print(f"P2 Gen {gen:3d}/{NUM_GEN_P2} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever_p2:.4f}  "
              f"σ={es_p2.current_sigma:.6f}  [{elapsed:.1f}s/gen, ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Phase 2 abschließen
selector_p2.set_flat_params(best_ever_params_p2)
total_time_p2 = time.time() - t_total_p2
print(f"\n✅ PHASE 2 abgeschlossen ({total_time_p2/60:.1f} min)")
print(f"   Best-ever LL: {best_ever_p2:.4f}")

# ── SPEICHERN: Final Model nach Phase 2 ──
model_state_final = {name: p.data.clone() for name, p in model.named_parameters()}
torch.save(model_state_final, "/content/model_final.pt")
print(f"\n📦 Final Model gespeichert: /content/model_final.pt")

print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE: Sequential 2-Phase")
print(f"  Phase 1: {total_time_p1/60:.1f} min (Attention [18-21])")
print(f"  Phase 2: {total_time_p2/60:.1f} min (MLP [21])")
print(f"  Total:   {(total_time_p1 + total_time_p2)/60:.1f} min")
print(f"{'='*70}")


In [ ]:
# ── Evaluation: Final Model (nach Phase 1 + Phase 2) vs Base-Modell ──

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200

print("\n" + "="*70)
print("  EVALUATION: Final Model vs Base-Modell")
print("="*70)

# --- 1) Final Model laden (aus Phase 2) ---
print(f"\n1. Final Model laden (Phase 2 State)...")
final_state = torch.load("/content/model_final.pt", weights_only=False)
for name, param in model.named_parameters():
    if name in final_state:
        param.data.copy_(final_state[name])
print(f"   ✅ Final Model geladen")

# --- 2) Final Model evaluieren ---
print(f"\n2. Evaluiere FINAL MODEL auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
final_acc = eval_fn.evaluate(model, tokenizer)
t_final = time.time() - t0
print(f"   Final Accuracy: {final_acc:.2%}  ({t_final:.0f}s)")

# --- 3) Base-Modell laden und evaluieren ---
print(f"\n3. Lade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"\n4. Evaluiere BASE-MODELL auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"   Base Accuracy:  {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 4) Ergebnis ---
diff = final_acc - base_acc
print(f"\n{'='*70}")
print(f"  ERGEBNISSE AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"  Training: TRAIN-Split | Evaluation: TEST-Split (kein Data Leakage)")
print(f"{'='*70}")
print(f"  Base-Modell:           {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Final (Phase 1+2):     {final_acc:6.2%}  ({int(final_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Verbesserung:          {diff:+6.2%}")
print(f"{'='*70}")


In [ ]:
# ── Visualisierung: Option D++ (Phase 1 + Phase 2) + Accuracy ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# ── Plot 1: Phase 1 Fitness ──
ax1 = fig.add_subplot(gs[0, 0])
gens_p1 = [h["gen"] for h in history_p1]
bests_p1 = [h["best"] for h in history_p1]
means_p1 = [h["mean"] for h in history_p1]
best_evers_p1 = [h["best_ever"] for h in history_p1]

ax1.plot(gens_p1, bests_p1, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens_p1, means_p1, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens_p1, best_evers_p1, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood")
ax1.set_title("Phase 1: Attention [18-21]\n(Context-Integration)")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Phase 2 Fitness ──
ax2 = fig.add_subplot(gs[0, 1])
gens_p2 = [h["gen"] for h in history_p2]
bests_p2 = [h["best"] for h in history_p2]
means_p2 = [h["mean"] for h in history_p2]
best_evers_p2 = [h["best_ever"] for h in history_p2]

ax2.plot(gens_p2, bests_p2, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#FF6F00")
ax2.plot(gens_p2, means_p2, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FFA726")
ax2.plot(gens_p2, best_evers_p2, label="Best-Ever", linewidth=2, color="#D32F2F", linestyle="--")
ax2.set_xlabel("Generation")
ax2.set_ylabel("Log-Likelihood")
ax2.set_title("Phase 2: MLP [21]\n(Faktenwissen)")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Combined Sigma-Schedule ──
ax3 = fig.add_subplot(gs[0, 2])
sigmas_p1 = [h["sigma"] for h in history_p1]
sigmas_p2 = [h["sigma"] for h in history_p2]

# Phase 1: gens_p1 normal, Phase 2: gens_p2 + len(gens_p1)
gens_combined = list(gens_p1) + [g + len(gens_p1) for g in gens_p2]
sigmas_combined = sigmas_p1 + sigmas_p2

ax3.plot(gens_p1, sigmas_p1, linewidth=2, color="#2196F3", label="Phase 1: σ 0.003→0.001")
ax3.plot([g + len(gens_p1) for g in gens_p2], sigmas_p2, linewidth=2, color="#D32F2F", label="Phase 2: σ 0.001→0.0005")
ax3.axvline(x=len(gens_p1), color="gray", linestyle=":", linewidth=2, alpha=0.5, label="Phase-Grenze")
ax3.set_xlabel("Generation (kombiniert)")
ax3.set_ylabel("σ (Noise Std)")
ax3.set_title("Cosine σ-Decay Schedule\n(beide Phasen)")
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=8)

# ── Plot 4: Accuracy-Vergleich ──
ax4 = fig.add_subplot(gs[1, 0:2])
labels = ["Base-Modell\n(unverändert)", "Final Model\n(Phase 1 + Phase 2)"]
accs = [base_acc, final_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax4.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=1, alpha=0.85)
for bar, acc in zip(bars, accs):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=12)

ax4.set_ylabel("Accuracy", fontsize=11)
ax4.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})\nTrain: train-Split | Eval: test-Split", fontsize=12)
ax4.set_ylim(0, max(accs) * 1.3)
ax4.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, linewidth=1)
ax4.grid(axis="y", alpha=0.3)

# ── Plot 5: Laufzeit Summary ──
ax5 = fig.add_subplot(gs[1, 2])
times_p1 = [h["elapsed"] for h in history_p1]
times_p2 = [h["elapsed"] for h in history_p2]

phases = ["Phase 1\n(Attention\n[18-21])", "Phase 2\n(MLP\n[21])"]
total_times = [sum(times_p1), sum(times_p2)]
colors_phases = ["#2196F3", "#D32F2F"]

bars_time = ax5.bar(phases, total_times, color=colors_phases, width=0.6, edgecolor="black", linewidth=1)
for bar, t in zip(bars_time, total_times):
    ax5.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
             f"{t/60:.1f} min\n({int(t/len(times_p1))}s/gen avg)",
             ha="center", va="bottom", fontweight="bold", fontsize=10)

ax5.set_ylabel("Total Time (seconds)", fontsize=10)
ax5.set_title("Training Duration\n(komplett)", fontsize=11)
ax5.axhline(y=sum(total_times)/2, color="gray", linestyle="--", alpha=0.3)
ax5.grid(axis="y", alpha=0.3)

fig.suptitle("Option D++: Sequential Fine-Tuning (2 Phasen)\nPhase 1: Attention [18-21] → Phase 2: MLP [21]",
             fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
print(f"\n{'='*70}")
print(f"  ZUSAMMENFASSUNG – Option D++ (Sequential 2-Phase)")
print(f"{'='*70}")
print(f"\n  PHASE 1: Attention [18-21]")
print(f"  ──────────────────────────")
print(f"  Generationen:      {len(history_p1)}")
print(f"  Population:        96")
print(f"  Parameter:         7,200,000 (Attention only)")
print(f"  σ-Schedule:        0.003 → 0.001 (cosine)")
print(f"  Best-Ever LL:      {best_ever_p1:.4f}")
print(f"  Total Zeit:        {sum(times_p1):.0f}s ({sum(times_p1)/60:.1f} min)")
print(f"  Ø Zeit/Gen:        {np.mean(times_p1):.1f}s")

print(f"\n  PHASE 2: MLP [21] (auf Phase-1-State)")
print(f"  ────────────────────────────────────")
print(f"  Generationen:      {len(history_p2)}")
print(f"  Population:        96")
print(f"  Parameter:         12,000,000 (MLP only [21])")
print(f"  σ-Schedule:        0.001 → 0.0005 (cosine, konservativer)")
print(f"  Best-Ever LL:      {best_ever_p2:.4f}")
print(f"  Total Zeit:        {sum(times_p2):.0f}s ({sum(times_p2)/60:.1f} min)")
print(f"  Ø Zeit/Gen:        {np.mean(times_p2):.1f}s")

total_combined = sum(times_p1) + sum(times_p2)
print(f"\n  GESAMT (Phase 1 + Phase 2)")
print(f"  ──────────────────────────")
print(f"  Total Generationen: {len(history_p1) + len(history_p2)}")
print(f"  Total Parameter:    19,200,000 (7.2M Attn + 12M MLP)")
print(f"  Total Zeit:         {total_combined:.0f}s ({total_combined/60:.1f} min)")

print(f"\n  ACCURACY RESULTS (TEST-Split, n={NUM_EVAL})")
print(f"  ──────────────────────────────────────────")
print(f"  Base-Modell:       {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Final Model:       {final_acc:6.2%}  ({int(final_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Verbesserung:      {diff:+6.2%}  ({diff*NUM_EVAL:+.0f} Fragen)")
print(f"{'='*70}")


In [ ]:
# ── Visualisierung: v5 Training + Accuracy-Vergleich ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# ── Plot 1: Fitness über Generationen ──
ax1 = fig.add_subplot(gs[0, 0])
gens = [h["gen"] for h in history]
bests = [h["best"] for h in history]
means = [h["mean"] for h in history]
best_evers = [h["best_ever"] for h in history]

ax1.plot(gens, bests, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens, means, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens, best_evers, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood Fitness")
ax1.set_title("ES Training v5 – Fitness über Generationen\n(4 Layers [18-21] + Cosine σ)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Sigma-Schedule ──
ax2 = fig.add_subplot(gs[0, 1])
sigmas = [h["sigma"] for h in history]
ax2.plot(gens, sigmas, linewidth=2, color="#9C27B0")
ax2.set_xlabel("Generation")
ax2.set_ylabel("σ (Noise Std)")
ax2.set_title("Cosine σ-Decay: 0.003 → 0.001")
ax2.grid(True, alpha=0.3)
ax2.fill_between(gens, sigmas, alpha=0.15, color="#9C27B0")

# ── Plot 3: Accuracy-Vergleich ──
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Base-Modell\n(unverändert)", "Fine-tuned v5\n(4 Layers, 100 Gen)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax3.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)

ax3.set_ylabel("Accuracy")
ax3.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})")
ax3.set_ylim(0, max(accs) * 1.3)
ax3.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, label="Baseline")
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9)

# ── Plot 4: Zeitverlauf pro Generation ──
ax4 = fig.add_subplot(gs[1, 1])
times = [h["elapsed"] for h in history]
ax4.plot(gens, times, linewidth=1.5, color="#FF5722", alpha=0.7)
ax4.axhline(y=np.mean(times), color="gray", linestyle="--", alpha=0.5,
            label=f"Ø {np.mean(times):.1f}s/gen")
ax4.set_xlabel("Generation")
ax4.set_ylabel("Zeit (Sekunden)")
ax4.set_title("Laufzeit pro Generation")
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

fig.suptitle("ES-Training v5 – 4 Layers [18-21] + Cosine σ + Full Reasoning",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
total_time = sum(times)
print(f"\n{'='*65}")
print(f"  ZUSAMMENFASSUNG – ES-Training v5")
print(f"{'='*65}")
print(f"  Generationen:      {len(history)}")
print(f"  Population:        {es.population_size}")
print(f"  Parameter:         {selector.num_target_elements:,} (Layer 18-21 Attention)")
print(f"  Gesamtzeit:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Ø pro Generation:  {np.mean(times):.1f}s")
print(f"  σ Start → Ende:   {history[0]['sigma']:.5f} → {history[-1]['sigma']:.5f}")
print(f"  Best-Ever LL:      {history[-1]['best_ever']:.4f}")
print(f"  Data Reshuffling:  ON (120 neue Fragen/Gen aus Pool von 5000)")
print(f"  Batch-Size:        12")

In [ ]:
# ── Vergleich: Fine-tuned vs Base-Modell auf GSM8K (Accuracy) ──
# Testet das gerade trainierte Modell gegen das unveränderte Base-Modell
# mit jeweils 100 Fragen aus dem TEST-Split.

from es_llm.fitness.gsm8k import GSM8KFitness
import torch, time
import matplotlib.pyplot as plt

NUM_EVAL = 100

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell fuer Vergleich...")
from es_llm.model.loader import load_model_and_tokenizer as load_base
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*50}")
print(f"  Base-Modell:     {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned:      {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:       {diff:+6.2%}")
print(f"{'='*50}")

# --- 4) Plot ---
labels = ["Base-Modell", "Fine-tuned\n(ES + LL)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)

# Werte auf die Balken schreiben
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
            ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"GSM8K Accuracy – Base vs Fine-tuned (n={NUM_EVAL})", fontsize=13)
ax.set_ylim(0, max(accs) * 1.25)
ax.axhline(y=base_acc, color="red", linestyle="--", alpha=0.4, label="Baseline")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Ergebnisse visualisieren

In [ ]:
import matplotlib.pyplot as plt
import json

# Falls history nicht im Speicher ist (z.B. nach Kernel-Neustart):
# → Aus training_log.jsonl laden (Option A) oder manuell definieren
if "history" not in dir():
    # Option A: Aus JSONL-Datei laden
    from pathlib import Path
    log_path = Path("/content/experiments/runs")
    # Neuesten Run finden
    if log_path.exists():
        runs = sorted(log_path.iterdir(), key=lambda p: p.name, reverse=True)
        for run in runs:
            jsonl = run / "training_log.jsonl"
            if jsonl.exists():
                with open(jsonl) as f:
                    history = [json.loads(l) for l in f]
                print(f"History geladen aus: {jsonl}")
                break
    if "history" not in dir():
        raise RuntimeError(
            "Variable 'history' nicht gefunden!\n"
            "Bitte zuerst eine Trainings-Zelle ausfuehren (Option A, B1 oder B2)."
        )

gens = [h.get("gen") or h.get("generation") for h in history]
bests = [h.get("best") or h.get("best_fitness") for h in history]
means = [h.get("mean") or h.get("mean_fitness") for h in history]

plt.figure(figsize=(10, 5))
plt.plot(gens, bests, label="Best Fitness", linewidth=2)
plt.plot(gens, means, label="Mean Fitness", alpha=0.7)
if "baseline_acc" in dir():
    plt.axhline(y=baseline_acc, color="red", linestyle="--", label="Baseline")
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title("ES Training – Fitness über Generationen")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fine-tuned Modell evaluieren

In [ ]:
# Evaluation auf dem Test-Split (größeres Subset)
eval_fitness = GSM8KFitness(
    num_samples=100,
    split="test",
    max_new_tokens=256,
)

final_acc = eval_fitness.evaluate(model, tokenizer)
print(f"\n📊 Ergebnisse:")
print(f"   Baseline (test, n=30):   {baseline_acc:.2%}")
print(f"   Fine-tuned (test, n=100): {final_acc:.2%}")
print(f"   Differenz:                {final_acc - baseline_acc:+.2%}")

## 8. Modell & Ergebnisse speichern

In [ ]:
# ── Checkpoint speichern ──
import json
from pathlib import Path

save_dir = Path("/content/experiments/final")
save_dir.mkdir(parents=True, exist_ok=True)

# Layer-Gewichte speichern
layer_state = {name: p.data.cpu().clone() for name, p in selector.get_target_params()}
torch.save(layer_state, save_dir / "best_layer_weights.pt")

# Komplettes Modell speichern
model.save_pretrained(save_dir / "model")
tokenizer.save_pretrained(save_dir / "model")

# Ergebnisse speichern
results = {
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "target_layers": selector.target_names,
    "num_target_params": selector.num_target_elements,
    "history": history,
}
(save_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))

print(f"✅ Gespeichert in: {save_dir}")
!ls -la {save_dir}

In [ ]:
# ── Optional: Nach Google Drive kopieren ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/es-llm-experiments"
!mkdir -p {DRIVE_DIR}
!cp -r {save_dir}/* {DRIVE_DIR}/
print(f"✅ Kopiert nach Google Drive: {DRIVE_DIR}")

## 9. Gespeichertes Modell laden (in neuer Session)

So lädst du ein zuvor gespeichertes ES-Modell:

In [ ]:
# ── Variante A: Komplettes gespeichertes Modell ──
# model, tokenizer = load_model_and_tokenizer(
#     model_name="/content/experiments/final/model",
#     dtype="float16",
#     device="cuda",
# )

# ── Variante B: Base-Modell + Layer-Gewichte ──
# model, tokenizer = load_model_and_tokenizer(
#     "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
# )
# state = torch.load("/content/experiments/final/best_layer_weights.pt", weights_only=True)
# model_dict = dict(model.named_parameters())
# for name, tensor in state.items():
#     model_dict[name].data.copy_(tensor.to(model.device))
# print(f"Loaded {len(state)} layer parameters")

---

## Tipps für Colab

| Thema | Empfehlung |
|-------|------------|
| **GPU-Typ** | T4 (free) reicht für Qwen 0.5B. A100 (Pro) für schnellere Inferenz |
| **dtype** | Immer `float16` auf GPU → halber Speicher, schneller |
| **VRAM** | Qwen 0.5B in fp16 ≈ 1 GB. T4 hat 16 GB → viel Headroom |
| **Timeout** | Colab trennt nach ~90min Inaktivität. Halte Tab offen |
| **Ergebnisse** | Immer nach Google Drive speichern – Colab ist flüchtig! |

### Konfiguration: Voller Layer (~14.9M Parameter)

| Einstellung | Quick-Test | Experiment |
|-------------|-----------|------------|
| **components** | `"all"` | `"all"` |
| **layer_indices** | [23] | [23] |
| **σ** | 0.001 | 0.001 |
| **learning_rate** | 0.001 | 0.001 |
| **population_size** | 10 | 20 |
| **num_samples** | 15 | 25 |
| **num_generations** | 10 | 50 |
| **Inferenzen gesamt** | ~1.500 | ~25.000 |

**Wichtig bei 14.9M Dimensionen:** σ muss **sehr klein** sein (0.001), weil eine Perturbation über 14.9M Parameter sonst das Modellverhalten zerstört.

## 10. Multi-Task Fine-Tuning: Minerva Algebra + HellaSwag

Dieser Block trainiert `Qwen/Qwen2.5-0.5B-Instruct` auf einer kombinierten Trainingsmenge aus Algebra- und HellaSwag-Beispielen und evaluiert danach mit OLMES auf:
- `minerva_math_algebra`
- `hellaswag`

Hinweis:
- Das ist ein pragmatisches SFT-Setup (LoRA) fuer schnelle Experimente.
- OLMES muss in der Runtime verfuegbar sein.

In [ ]:
# Abhaengigkeiten fuer Multi-Task-Finetuning (hard reset)
# Fix fuer gemischte transformers-Installationen in Colab

import site
import glob
import shutil
from pathlib import Path

# 1) Alte Paketordner physisch entfernen (wichtig bei inkonsistenter Installation)
site_paths = site.getsitepackages() + [site.getusersitepackages()]
patterns = [
    "transformers*",
    "tokenizers*",
    "accelerate*",
    "peft*",
    "datasets*",
    "huggingface_hub*",
]

for sp in site_paths:
    for pat in patterns:
        for p in glob.glob(str(Path(sp) / pat)):
            try:
                if Path(p).is_dir():
                    shutil.rmtree(p, ignore_errors=True)
                else:
                    Path(p).unlink(missing_ok=True)
            except Exception:
                pass

# 2) Deinstallieren + sauber neu installieren
%pip uninstall -y -q transformers tokenizers accelerate peft datasets huggingface_hub deepspeed
%pip install -q --no-cache-dir --upgrade --force-reinstall \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "datasets==3.1.0" \
    "huggingface_hub==0.26.2"

In [ ]:
# Versionen pruefen (ohne direkten Modulimport) + Trainer-Smoketest
import importlib.metadata as im

for pkg in ["transformers", "accelerate", "peft", "datasets", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{pkg}: {im.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: NOT FOUND ({e})")

print("\nWenn du gerade Cell 49 ausgefuehrt hast: Runtime jetzt NEU STARTEN und dann erst weiter.")

# Diesen Block erst NACH Neustart ausfuehren:
# from transformers import Trainer, TrainingArguments
# print("Trainer import OK")

In [ ]:
import os
import json
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_BASE = Path("/content/experiments/minerva_hellaswag")
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
FT_OUT = OUTPUT_BASE / f"ft_{RUN_TAG}"
EVAL_OUT = OUTPUT_BASE / f"olmes_eval_{RUN_TAG}"

MAX_LEN = 384
N_ALG_TRAIN = 4000
N_HELLA_TRAIN = 4000
N_ALG_VAL = 400
N_HELLA_VAL = 400

print("FT_OUT:", FT_OUT)
print("EVAL_OUT:", EVAL_OUT)
print("CUDA:", torch.cuda.is_available())

In [ ]:
# 1) Datensaetze laden und in gemeinsames SFT-Format bringen
# Algebra: zuerst hendrycks/competition_math versuchen, dann Fallback EleutherAI/hendrycks_math
# HellaSwag: Rowan/hellaswag

try:
    alg_raw = load_dataset("hendrycks/competition_math", "algebra")
    print("Loaded algebra dataset: hendrycks/competition_math")
except Exception:
    alg_raw = load_dataset("EleutherAI/hendrycks_math", "algebra")
    print("Loaded algebra dataset: EleutherAI/hendrycks_math")

hella_raw = load_dataset("Rowan/hellaswag")


def format_algebra(ex):
    prompt = (
        "You are a math tutor. Solve the algebra problem step by step.\n"
        f"Problem: {ex['problem']}\n"
        "Answer:"
    )
    target = ex.get("solution", "")
    text = f"{prompt} {target}"
    return {"text": text, "source_task": "algebra"}


def format_hellaswag(ex):
    label = int(ex["label"])
    ending = ex["endings"][label]
    prompt = (
        "Choose the most plausible continuation for the context and explain briefly.\n"
        f"Context: {ex['ctx']}\n"
        "Best continuation:"
    )
    text = f"{prompt} {ending}"
    return {"text": text, "source_task": "hellaswag"}

alg_train = alg_raw["train"].shuffle(seed=SEED).select(range(min(N_ALG_TRAIN, len(alg_raw["train"]))))
alg_val = alg_raw["test"].shuffle(seed=SEED).select(range(min(N_ALG_VAL, len(alg_raw["test"]))))

hella_train = hella_raw["train"].shuffle(seed=SEED).select(range(min(N_HELLA_TRAIN, len(hella_raw["train"]))))
hella_val = hella_raw["validation"].shuffle(seed=SEED).select(range(min(N_HELLA_VAL, len(hella_raw["validation"]))))

alg_train = alg_train.map(format_algebra, remove_columns=alg_train.column_names)
alg_val = alg_val.map(format_algebra, remove_columns=alg_val.column_names)
hella_train = hella_train.map(format_hellaswag, remove_columns=hella_train.column_names)
hella_val = hella_val.map(format_hellaswag, remove_columns=hella_val.column_names)

train_ds = concatenate_datasets([alg_train, hella_train]).shuffle(seed=SEED)
val_ds = concatenate_datasets([alg_val, hella_val]).shuffle(seed=SEED)

print("Train size:", len(train_ds), "| Val size:", len(val_ds))
print("Sample:\n", train_ds[0]["text"][:500])

In [ ]:
# 2) Tokenisierung + LoRA-Modell aufsetzen

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()


def tokenize_batch(batch):
    tok = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Tokenized train size:", len(train_tok), "| val size:", len(val_tok))

In [ ]:
# 3) Training starten (manueller Loop, ohne Trainer)

FT_OUT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

train_tok = train_tok.with_format("torch")
val_tok = val_tok.with_format("torch")

train_loader = DataLoader(
    train_tok,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)
val_loader = DataLoader(
    val_tok,
    batch_size=2,
    shuffle=False,
    collate_fn=collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
num_epochs = 1
grad_acc_steps = 8
max_steps = num_epochs * max(1, len(train_loader) // grad_acc_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.03 * max_steps)),
    num_training_steps=max(1, max_steps),
)

model.train()
global_step = 0
running_loss = 0.0

for epoch in range(num_epochs):
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / grad_acc_steps
        loss.backward()
        running_loss += loss.item()

        if step % grad_acc_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % 20 == 0:
                print(f"epoch={epoch+1} step={global_step} train_loss={running_loss/20:.4f}")
                running_loss = 0.0

# einfache Val-Loss Messung
model.eval()
val_losses = []
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 100:
            break
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        val_losses.append(float(out.loss.item()))

if val_losses:
    print(f"Validation loss (sampled): {sum(val_losses)/len(val_losses):.4f}")

model.save_pretrained(str(FT_OUT / "model"))
tokenizer.save_pretrained(str(FT_OUT / "model"))

print("Fine-tuned model saved to:", FT_OUT / "model")

In [ ]:
# 4) OLMES Evaluation auf minerva_math_algebra + hellaswag
# Nutzt das Projektskript scripts/eval_olmes_tasks.py

import shutil
import subprocess

if shutil.which("olmes") is None:
    raise RuntimeError(
        "`olmes` ist nicht im PATH. Bitte in dieser Runtime installieren oder im Projekt-Container ausfuehren."
    )

EVAL_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    "python3", "scripts/eval_olmes_tasks.py",
    "--model", str(FT_OUT / "model"),
    "--tasks", "minerva_math_algebra", "hellaswag",
    "--model_type", "hf",
    "--limit", "300",
    "--num_shots", "0",
    "--batch_size", "2",
    "--cache_dir", "cache",
    "--output_dir", str(EVAL_OUT),
]

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"OLMES Eval failed with code {proc.returncode}")

print("OLMES output:", EVAL_OUT)

In [ ]:
# 5) Metriken laden und visualisieren

import pandas as pd
import matplotlib.pyplot as plt


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


metrics_path = EVAL_OUT / "metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"metrics.json nicht gefunden in {EVAL_OUT}")

metrics = read_json(metrics_path)
rows = []
for t in metrics.get("tasks", []):
    m = t.get("metrics", {}) or {}
    rows.append({
        "task": t.get("alias") or t.get("task_name"),
        "primary_score": m.get("primary_score"),
        "num_instances": t.get("num_instances"),
        "processing_time_sec": t.get("processing_time"),
        "acc_raw": m.get("acc_raw"),
        "acc_norm": m.get("acc_norm"),
        "exact_match": m.get("exact_match"),
        "exact_match_flex": m.get("exact_match_flex"),
    })

metrics_df = pd.DataFrame(rows)
display(metrics_df)

chart_df = metrics_df.dropna(subset=["primary_score"]).copy()
if not chart_df.empty:
    plt.figure(figsize=(8, 4))
    plt.bar(chart_df["task"], chart_df["primary_score"])
    plt.title("OLMES Primary Score pro Task")
    plt.ylabel("score")
    plt.xlabel("task")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6) Fehlbeispiele und Token-Verteilung analysieren

pred_rows = []
for pf in sorted(EVAL_OUT.glob("task-*-predictions.jsonl")):
    parts = pf.name.split("-")
    task_alias = parts[2] if len(parts) >= 4 else "unknown"
    for r in read_jsonl(pf):
        mo = (r.get("model_output") or [{}])[0] or {}
        mt = r.get("metrics") or {}
        pred_rows.append({
            "task": task_alias,
            "doc_id": r.get("doc_id"),
            "gold": r.get("label"),
            "model_answer": mo.get("model_answer") or mo.get("continuation"),
            "exact_match": mt.get("exact_match"),
            "exact_match_flex": mt.get("exact_match_flex"),
            "num_tokens": mt.get("num_tokens"),
            "max_tokens_reached": mt.get("max_tokens_reached"),
        })

pred_df = pd.DataFrame(pred_rows)
print("Pred rows:", len(pred_df))

if not pred_df.empty:
    score_col = "exact_match_flex" if pred_df["exact_match_flex"].notna().any() else "exact_match"
    pred_df["is_error"] = pred_df[score_col].fillna(0).eq(0)

    err = pred_df.groupby("task", as_index=False)["is_error"].mean().rename(columns={"is_error": "error_rate"})
    display(err)

    plt.figure(figsize=(8, 4))
    plt.bar(err["task"], err["error_rate"])
    plt.title(f"Fehlerquote pro Task ({score_col})")
    plt.ylim(0, 1)
    plt.ylabel("error rate")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    tok = pred_df.dropna(subset=["num_tokens"])
    if not tok.empty:
        plt.figure(figsize=(8, 4))
        for tname, g in tok.groupby("task"):
            plt.hist(g["num_tokens"], bins=20, alpha=0.45, label=tname)
        plt.title("Token-Verteilung pro Task")
        plt.xlabel("num_tokens")
        plt.ylabel("count")
        plt.legend()
        plt.tight_layout()
        plt.show()

    display(
        pred_df[pred_df["is_error"]]
        .sort_values(["task", "num_tokens"], ascending=[True, False])
        .head(20)
    )
else:
    print("Keine predictions.jsonl gefunden.")

### 10.1 ES auf Original-Layern (ohne LoRA) fuer Minerva Algebra + HellaSwag

Dieser Teil trainiert direkt auf den Original-Layern via ES (keine Adapter) und dient als Vergleich gegen LoRA.

Ablauf:
1. Basismodell laden
2. Layer selektieren
3. Multi-Task-Fitness auf Algebra + HellaSwag berechnen
4. ES-Optimierung
5. Modell speichern fuer OLMES-Eval

In [ ]:
import math
import re
from typing import List

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES

ES_OUT = OUTPUT_BASE / f"es_layer_{RUN_TAG}"
ES_OUT.mkdir(parents=True, exist_ok=True)

# Kleines, stabiles Fitness-Subset fuer ES (schneller als OLMES-in-the-loop)
ALG_ES_N = 80
HELLA_ES_N = 120

alg_es = alg_raw["train"].shuffle(seed=SEED).select(range(min(ALG_ES_N, len(alg_raw["train"]))))
hella_es = hella_raw["train"].shuffle(seed=SEED).select(range(min(HELLA_ES_N, len(hella_raw["train"]))))


def _extract_last_number(text: str):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None


def _avg_logprob_for_suffix(model, tokenizer, prompt: str, suffix: str) -> float:
    full = prompt + suffix
    tok_full = tokenizer(full, return_tensors="pt").to(model.device)
    tok_prompt = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model(**tok_full)
        logits = out.logits[:, :-1, :]
        labels = tok_full["input_ids"][:, 1:]
        log_probs = torch.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    prompt_len = tok_prompt["input_ids"].shape[1] - 1
    suffix_lp = tok_lp[:, prompt_len:]
    if suffix_lp.numel() == 0:
        return -1e9
    return float(suffix_lp.mean().item())


def es_multitask_fitness(model, tokenizer) -> float:
    model.eval()

    # Algebra: einfache numerische Trefferquote
    alg_correct = 0
    for ex in alg_es:
        prompt = (
            "Solve the algebra problem step by step and output final numeric answer.\n"
            f"Problem: {ex['problem']}\nAnswer:"
        )
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        out_ids = gen[0, inputs["input_ids"].shape[1]:]
        pred_text = tokenizer.decode(out_ids, skip_special_tokens=True)

        pred = _extract_last_number(pred_text)
        gold = _extract_last_number(ex.get("solution", ""))
        alg_correct += int(pred is not None and gold is not None and pred == gold)

    alg_acc = alg_correct / max(1, len(alg_es))

    # HellaSwag: Kandidat mit hoechster mittlerer Log-Wahrscheinlichkeit
    hella_correct = 0
    for ex in hella_es:
        prompt = f"Context: {ex['ctx']}\nBest continuation: "
        scores = []
        for end in ex["endings"]:
            score = _avg_logprob_for_suffix(model, tokenizer, prompt, end)
            scores.append(score)
        pred_idx = int(max(range(len(scores)), key=lambda i: scores[i]))
        hella_correct += int(pred_idx == int(ex["label"]))

    hella_acc = hella_correct / max(1, len(hella_es))

    # Kombinierte Fitness
    return 0.5 * alg_acc + 0.5 * hella_acc

print("ES fitness subsets:", len(alg_es), "algebra |", len(hella_es), "hellaswag")

In [ ]:
# ES-Training auf Original-Layern

es_model, es_tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    dtype="float16" if torch.cuda.is_available() else "float32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Fokus auf spaete Layer, damit Compute handhabbar bleibt
es_selector = LayerSelector(
    model=es_model,
    strategy="by_index",
    layer_indices=[22, 23],
    components="attention_qkv",
)
print(es_selector.summary())

es_algo = OpenAIES(
    population_size=12,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    seed=SEED,
)

ES_GENERATIONS = 6
current_params = es_selector.get_flat_params().clone()
best_params = current_params.clone()

baseline_fit = es_multitask_fitness(es_model, es_tokenizer)
best_fit = baseline_fit
es_history = [{"gen": 0, "best": baseline_fit, "mean": baseline_fit}]
print(f"Baseline multi-task fitness: {baseline_fit:.4f}")

for gen in range(1, ES_GENERATIONS + 1):
    candidates = es_algo.ask(current_params)
    fits = []
    for cand in candidates:
        es_selector.set_flat_params(cand)
        fit = es_multitask_fitness(es_model, es_tokenizer)
        fits.append(fit)

    result = es_algo.tell(current_params, candidates, fits)
    current_params = result.new_params.clone()
    es_selector.set_flat_params(current_params)

    if result.best_fitness > best_fit:
        best_fit = result.best_fitness
        best_params = current_params.clone()

    es_history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"ES Gen {gen:02d}/{ES_GENERATIONS} | best={result.best_fitness:.4f} | mean={result.mean_fitness:.4f} | best_ever={best_fit:.4f}")

# Besten Zustand anwenden und speichern
es_selector.set_flat_params(best_params)
es_model.save_pretrained(ES_OUT / "model")
es_tokenizer.save_pretrained(ES_OUT / "model")

with open(ES_OUT / "es_history.json", "w", encoding="utf-8") as f:
    json.dump(es_history, f, indent=2)

print("ES model saved to:", ES_OUT / "model")

In [ ]:
# 10.2 OLMES-Eval: Base vs ES Layer vs LoRA

import shutil
import subprocess

if shutil.which("olmes") is None:
    raise RuntimeError("`olmes` ist nicht im PATH. Bitte OLMES in der Runtime verfuegbar machen.")

COMPARE_OUT = OUTPUT_BASE / f"compare_{RUN_TAG}"
COMPARE_OUT.mkdir(parents=True, exist_ok=True)


def run_olmes_for_model(model_tag: str, model_ref: str, limit: int = 300) -> Path:
    out_dir = COMPARE_OUT / model_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python3", "scripts/eval_olmes_tasks.py",
        "--model", model_ref,
        "--tasks", "minerva_math_algebra", "hellaswag",
        "--model_type", "hf",
        "--limit", str(limit),
        "--num_shots", "0",
        "--batch_size", "2",
        "--cache_dir", "cache",
        "--output_dir", str(out_dir),
    ]
    print("Running", model_tag, ":", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"OLMES failed for {model_tag} with code {proc.returncode}")
    return out_dir

base_out = run_olmes_for_model("base", MODEL_NAME)
es_out = run_olmes_for_model("es_layer", str(ES_OUT / "model"))
lora_out = run_olmes_for_model("lora", str(FT_OUT / "model"))

print("Compare outputs:", base_out, es_out, lora_out)

In [ ]:
# 10.3 Vergleich visualisieren

import pandas as pd
import matplotlib.pyplot as plt


def load_metrics_rows(run_dir: Path, model_tag: str):
    p = run_dir / "metrics.json"
    if not p.exists():
        return []
    with p.open("r", encoding="utf-8") as f:
        m = json.load(f)

    rows = []
    for t in m.get("tasks", []):
        tm = t.get("metrics", {}) or {}
        rows.append({
            "model": model_tag,
            "task": t.get("alias") or t.get("task_name"),
            "primary_score": tm.get("primary_score"),
            "acc_raw": tm.get("acc_raw"),
            "acc_norm": tm.get("acc_norm"),
            "exact_match": tm.get("exact_match"),
            "exact_match_flex": tm.get("exact_match_flex"),
        })
    return rows

compare_rows = []
compare_rows += load_metrics_rows(base_out, "Base")
compare_rows += load_metrics_rows(es_out, "ES Layer")
compare_rows += load_metrics_rows(lora_out, "LoRA")

compare_df = pd.DataFrame(compare_rows)
display(compare_df.sort_values(["task", "model"]))

plot_df = compare_df.dropna(subset=["primary_score"]).copy()
if not plot_df.empty:
    pivot = plot_df.pivot(index="task", columns="model", values="primary_score")
    display(pivot)

    ax = pivot.plot(kind="bar", figsize=(9, 5))
    ax.set_title("OLMES Vergleich: Base vs ES Layer vs LoRA")
    ax.set_ylabel("primary_score")
    ax.set_xlabel("task")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Keine primary_score-Werte gefunden.")